# 한국 소설 RAG - 인덱싱 파이프라인

**파이프라인**
```
Azure Blob Storage (novels_raw.json)
→ Azure OpenAI text-embedding-3-small (임베딩)
→ Azure AI Search (하이브리드 인덱스 생성)
```


In [ ]:
!pip install azure-search-documents azure-storage-blob openai python-dotenv tqdm -q

In [ ]:
import os, json, time
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.storage.blob import BlobServiceClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    SearchFieldDataType, VectorSearch, HnswAlgorithmConfiguration,
    VectorSearchProfile, SemanticConfiguration, SemanticSearch,
    SemanticPrioritizedFields, SemanticField
)
from azure.core.credentials import AzureKeyCredential
from tqdm import tqdm

load_dotenv(override=True)
print("✅ 라이브러리 로드 완료")


## 1. 설정

In [ ]:
STORAGE_CONN   = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
CONTAINER_NAME = os.getenv("AZURE_STORAGE_CONTAINER", "novels")
SEARCH_EP      = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_KEY     = os.getenv("AZURE_SEARCH_ADMIN_KEY")
OPENAI_EP      = os.getenv("AZURE_OPENAI_ENDPOINT")
OPENAI_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
EMBED_DEPLOY   = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
INDEX_NAME     = "korean-novels"

for name, val in {
    "AZURE_STORAGE_CONNECTION_STRING": STORAGE_CONN,
    "AZURE_SEARCH_ENDPOINT":           SEARCH_EP,
    "AZURE_SEARCH_ADMIN_KEY":          SEARCH_KEY,
    "AZURE_OPENAI_ENDPOINT":           OPENAI_EP,
    "AZURE_OPENAI_API_KEY":            OPENAI_KEY,
    "EMBED_DEPLOYMENT":                EMBED_DEPLOY,
}.items():
    print(f"{'✅' if val else '❌ None'} {name}")

openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_EP,
    api_key=OPENAI_KEY,
    api_version="2024-02-01"
)
index_client = SearchIndexClient(
    endpoint=SEARCH_EP,
    credential=AzureKeyCredential(SEARCH_KEY)
)
search_client = SearchClient(
    endpoint=SEARCH_EP,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY)
)
print("\n✅ 클라이언트 초기화 완료")


## 2. Blob에서 데이터 로드

In [ ]:
blob_service = BlobServiceClient.from_connection_string(STORAGE_CONN)
blob_client = blob_service.get_blob_client(
    container=CONTAINER_NAME,
    blob="novels_raw.json"
)

data = blob_client.download_blob().readall()
novels = json.loads(data)

print(f"✅ Blob에서 로드 완료: {len(novels)}건")
print("\n샘플 1건:")
print(json.dumps(novels[0], ensure_ascii=False, indent=2))


## 3. 임베딩 생성

In [ ]:
def get_embedding(text: str) -> list[float]:
    text = str(text).strip() or "내용 없음"
    response = openai_client.embeddings.create(
        input=text,
        model=EMBED_DEPLOY
    )
    return response.data[0].embedding


# summary + category 합쳐서 임베딩
print(f"임베딩 생성 중... ({len(novels)}건)")
for novel in tqdm(novels):
    text = f"{novel.get('category', '')} {novel.get('summary', '')}".strip()
    novel["summary_vector"] = get_embedding(text)
    time.sleep(0.05)  # RateLimit 방지

print(f"✅ 임베딩 완료 | 차원: {len(novels[0]['summary_vector'])}")


## 4. Azure AI Search 인덱스 생성

In [ ]:
VECTOR_DIM = len(novels[0]["summary_vector"])

fields = [
    SimpleField(name="id",           type=SearchFieldDataType.String, key=True),
    SearchableField(name="title",    type=SearchFieldDataType.String, analyzer_name="ko.microsoft"),
    SearchableField(name="author",   type=SearchFieldDataType.String),
    SimpleField(name="publisher",    type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="pub_year",     type=SearchFieldDataType.String, filterable=True, sortable=True),
    SimpleField(name="category",     type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="summary",  type=SearchFieldDataType.String, analyzer_name="ko.microsoft"),
    SimpleField(name="isbn",         type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="img_url",      type=SearchFieldDataType.String),
    SimpleField(name="page_count",   type=SearchFieldDataType.Int32,  filterable=True, sortable=True),
    SearchField(
        name="summary_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=VECTOR_DIM,
        vector_search_profile_name="hnsw-profile"
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-algo")],
    profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-algo")]
)

semantic_search = SemanticSearch(configurations=[
    SemanticConfiguration(
        name="semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[SemanticField(field_name="summary")],
            keywords_fields=[SemanticField(field_name="category")]
        )
    )
])

try:
    index_client.delete_index(INDEX_NAME)
    print("기존 인덱스 삭제")
except:
    pass

index_client.create_index(SearchIndex(
    name=INDEX_NAME, fields=fields,
    vector_search=vector_search, semantic_search=semantic_search
))
print(f"✅ 인덱스 [{INDEX_NAME}] 생성 완료")


## 5. 문서 업로드

In [ ]:
# id 필드 정리 (빈 값 방지)
for i, n in enumerate(novels):
    if not n.get("id"):
        n["id"] = f"novel_{i:05d}"

# 100건씩 배치 업로드
batch_size = 100
total_success = 0

for i in tqdm(range(0, len(novels), batch_size), desc="업로드"):
    batch = novels[i:i+batch_size]
    results = search_client.upload_documents(documents=batch)
    success = sum(1 for r in results if r.succeeded)
    total_success += success

print(f"\n✅ 업로드 완료: {total_success}/{len(novels)}건")


## 6. 검색 테스트

In [ ]:
from azure.search.documents.models import VectorizedQuery

def hybrid_search(query: str, top_k: int = 3) -> list[dict]:
    vector_query = VectorizedQuery(
        vector=get_embedding(query),
        k_nearest_neighbors=top_k,
        fields="summary_vector"
    )
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["title", "author", "pub_year", "category", "summary", "page_count", "img_url"],
        top=top_k
    )
    return [dict(r) for r in results]


# 테스트
for query in ["가족 갈등 현대소설", "역사 배경 대하소설", "따뜻한 힐링 소설"]:
    print(f"\n🔍 [{query}]")
    for r in hybrid_search(query):
        print(f"  {r['title']} - {r['author']} ({r['pub_year']}) | {r['page_count']}p")
